# HydroSense-Kenya – Level 4: Data Analysis, Cleaning & Visualization
**ICS 2207 Scientific Computing | February – May 2026**
---

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from data_cleaning import (load_datasets, detect_outliers_iqr,
                            clean_weather, clean_soil, save_cleaned)

plt.rcParams.update({'figure.dpi':120,'axes.grid':True,'grid.alpha':0.3,
                     'axes.spines.top':False,'axes.spines.right':False,'font.size':11})

weather_raw, soil_raw, params = load_datasets('../data/raw')
print("Raw datasets loaded.")
print(f"  weather: {weather_raw.shape}  |  soil: {soil_raw.shape}  |  params: {params.shape}")

## 2. Identify Data Quality Issues

In [ ]:
print("=== WEATHER: Missing values ===")
print(weather_raw.isnull().sum())

print("\n=== WEATHER: Descriptive statistics ===")
print(weather_raw.describe().round(3))

print("\n=== SOIL: Missing values ===")
print(soil_raw.isnull().sum())

print("\n=== SOIL: Unique sensor statuses ===")
print(soil_raw['sensor_status'].value_counts())

print("\n=== SOIL: Min/Max soil moisture ===")
print(soil_raw.groupby('zone_id')['soil_moisture_pct'].agg(['min','max','mean','std']).round(3))

In [ ]:
# Identify all outliers before cleaning
print("=== Outlier Detection (IQR method) ===\n")
for col in ['rainfall_mm','temperature_c']:
    mask = detect_outliers_iqr(weather_raw[col].dropna())
    outlier_vals = weather_raw[col].dropna()[mask]
    print(f"weather/{col}: {mask.sum()} outlier(s)")
    for idx, val in outlier_vals.items():
        print(f"   Row {idx} ({weather_raw.loc[idx,'date'].date()}): {val}")

print()
for col in ['soil_moisture_pct','tank_level_liters']:
    mask = detect_outliers_iqr(soil_raw[col].dropna())
    outlier_vals = soil_raw[col].dropna()[mask]
    print(f"soil/{col}: {mask.sum()} outlier(s)")
    for idx, val in outlier_vals.items():
        ts = soil_raw.loc[idx,'timestamp']
        zone = soil_raw.loc[idx,'zone_id']
        print(f"   Row {idx} ({ts}, {zone}): {val}")

## 3. Clean the Datasets

In [ ]:
print("=== Cleaning weather_daily.csv ===")
weather = clean_weather(weather_raw.copy())

print("\n=== Cleaning soil_sensor_data.csv ===")
soil = clean_soil(soil_raw.copy())

print("\n=== Post-cleaning checks ===")
print(f"Weather NAs remaining: {weather.isnull().sum().sum()}")
print(f"Soil NAs remaining:    {soil[['soil_moisture_pct','tank_level_liters']].isnull().sum().sum()}")

In [ ]:
# Recompute ET on cleaned weather
weather['et_mm'] = np.maximum(0, 0.12*weather['temperature_c'] + 0.35*weather['wind_speed_mps']
                                + 2.4*weather['solar_index'] - 0.025*weather['humidity_pct'])

# Save cleaned dataset
merged = save_cleaned(weather, soil, '../data/processed/cleaned_irrigation_dataset.csv')
print(merged.head(3).to_string())

## 4. Descriptive Statistics

In [ ]:
print("=== Weather – Descriptive Statistics (cleaned) ===")
print(weather[['rainfall_mm','temperature_c','humidity_pct','wind_speed_mps','solar_index','et_mm']].describe().round(3))

print("\n=== Soil Moisture by Zone ===")
print(soil.groupby('zone_id')['soil_moisture_pct'].agg(['mean','std','min','max']).round(3))

print("\n=== Pump Energy by Zone ===")
print(soil.groupby('zone_id')['pump_power_watts'].agg(['mean','sum']).round(1))

print(f"\nTotal rainfall (March 2026): {weather['rainfall_mm'].sum():.1f} mm (outlier-aware)")
print(f"Mean daily ET:               {weather['et_mm'].mean():.3f} mm/day")
print(f"Total ET (30 days):          {weather['et_mm'].sum():.2f} mm")

## 5. Scientific Visualisations

In [ ]:
# Figure 1: Rainfall & ET (dual axis)
fig, ax1 = plt.subplots(figsize=(12,5))
ax2 = ax1.twinx()
ax1.bar(weather['date'], weather['rainfall_mm'], color='steelblue', alpha=0.6, label='Rainfall (mm)')
ax2.plot(weather['date'], weather['et_mm'], 'o-', color='firebrick', lw=2, ms=4, label='ET (mm/day)')
ax1.set_xlabel('Date'); ax1.set_ylabel('Rainfall (mm)', color='steelblue')
ax2.set_ylabel('ET (mm/day)', color='firebrick')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0)); plt.xticks(rotation=30)
l1h,l1l=ax1.get_legend_handles_labels(); l2h,l2l=ax2.get_legend_handles_labels()
ax1.legend(l1h+l2h, l1l+l2l, loc='upper left')
plt.title('Figure 1: Daily Rainfall & ET – March 2026 (Cleaned)', fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figA1_rainfall_et.png', dpi=150, bbox_inches='tight'); plt.show()
print("Interpretation: ET is stable at 3-5 mm/day. The heavy rainfall on 19-20 Mar and the")
print("corrected 26 Mar event refill soil reserves. The 14 Mar temperature outlier is removed.")

In [ ]:
# Figure 2: Soil moisture by zone
ZONE_COLORS = {'Zone_A':'#e07b39','Zone_B':'#3a7ebf','Zone_C':'#5aab61'}
ZONE_CROPS  = {'Zone_A':'Tomato', 'Zone_B':'Kale',   'Zone_C':'Maize'}
fig, ax = plt.subplots(figsize=(12,5))
for zone_id, grp in soil.groupby('zone_id'):
    grp = grp.sort_values('timestamp')
    ax.plot(grp['timestamp'], grp['soil_moisture_pct'],
            color=ZONE_COLORS[zone_id], lw=2, marker='o', ms=3,
            label=f"{zone_id} ({ZONE_CROPS[zone_id]})")
for _, row in params.iterrows():
    ax.axhline(row['min_moisture_pct'], color=ZONE_COLORS[row['zone_id']], ls=':', alpha=0.6, lw=1.2,
               label=f"{row['zone_id']} stress threshold ({row['min_moisture_pct']}%)")
ax.set_xlabel('Date'); ax.set_ylabel('Soil Moisture (%)'); ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0)); plt.xticks(rotation=30)
plt.title('Figure 2: Soil Moisture Trends by Zone – March 2026', fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figA2_soil_moisture.png', dpi=150, bbox_inches='tight'); plt.show()
print("Interpretation: Zone_C (Maize) consistently has the lowest soil moisture and breaches")
print("its 20% threshold in late March, confirming it is the highest-priority irrigation zone.")

In [ ]:
# Figure 3: Boxplots of soil moisture by zone
fig, ax = plt.subplots(figsize=(8,5))
data_by_zone = [soil[soil['zone_id']==z]['soil_moisture_pct'].dropna().values for z in ['Zone_A','Zone_B','Zone_C']]
bp = ax.boxplot(data_by_zone, patch_artist=True, labels=['Zone_A\n(Tomato)','Zone_B\n(Kale)','Zone_C\n(Maize)'])
for patch, color in zip(bp['boxes'], ['#e07b39','#3a7ebf','#5aab61']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for _, row in params.iterrows():
    zone_idx = ['Zone_A','Zone_B','Zone_C'].index(row['zone_id']) + 1
    ax.plot(zone_idx, row['min_moisture_pct'], 'v', color='red', ms=10, zorder=5)
ax.set_ylabel('Soil Moisture (%)'); ax.set_title('Figure 3: Soil Moisture Distribution by Zone (▼ = stress threshold)', fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figA3_boxplots.png', dpi=150, bbox_inches='tight'); plt.show()
print("Interpretation: Zone_B has the highest variability (wider IQR). Zone_C median is closest")
print("to its stress threshold, confirming highest water-stress risk.")

In [ ]:
# Figure 4: Tank level trend
fig, ax = plt.subplots(figsize=(12,4))
for zone_id, grp in soil.groupby('zone_id'):
    grp = grp.sort_values('timestamp')
    ax.plot(grp['timestamp'], grp['tank_level_liters'],
            color=ZONE_COLORS[zone_id], lw=1.5, label=zone_id)
ax.set_xlabel('Date'); ax.set_ylabel('Tank Level (L)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0)); plt.xticks(rotation=30)
ax.legend(); ax.set_title('Figure 4: Irrigation Tank Level – March 2026', fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figA4_tank_level.png', dpi=150, bbox_inches='tight'); plt.show()
print("Interpretation: All zones show a steady drawdown of tank water. Zone_C draws down fastest")
print("due to its larger area and higher pump flow. Refill events are visible as sharp increases.")

In [ ]:
# Figure 5: Correlation heatmap (weather variables vs ET)
corr_cols = ['rainfall_mm','temperature_c','humidity_pct','wind_speed_mps','solar_index','et_mm']
corr = weather[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols))); ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha='center', va='center', fontsize=8,
                color='white' if abs(corr.iloc[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, label='Pearson r')
plt.title('Figure 5: Weather Variable Correlation Matrix', fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figA5_correlation.png', dpi=150, bbox_inches='tight'); plt.show()
print("Interpretation: ET correlates strongly with temperature (+) and solar_index (+),")
print("and negatively with humidity (-). Rainfall shows low correlation with ET,")
print("confirming they are independent drivers of the soil water balance.")

## 6. Summary
| Deliverable | Status |
|---|---|
| All datasets loaded with Pandas | ✅ |
| Missing values, outliers, anomalies identified | ✅ |
| Cleaning decisions documented | ✅ |
| Descriptive statistics computed | ✅ |
| 5 scientific visualisations with interpretations | ✅ |
| Cleaned dataset saved to data/processed/ | ✅ |

**Cleaning decisions:**
- `rainfall_mm` NA → 0 (missing sensor = no rain logged)
- `humidity_pct` NA → column median
- `temperature_c` 45.8°C on 14 Mar → replaced with median (physically implausible)
- `rainfall_mm` 85 mm on 26 Mar → flagged but retained (extreme but possible)
- `soil_moisture_pct` 8.5% Zone_B 25 Mar → linear interpolation (sensor fault)
- `tank_level_liters` 9900 L Zone_C 14 Mar → interpolated (tank capacity ~5000 L)
- `pump_flow_lpm` 0.0 Zone_B 21 Mar → flagged as pump fault (sensor_status = CHECK)